# 5 · unspecified-goal kv-transfer — algorithm swap with an un-named sub-goal

The usual kv-transfer specifies every sub-goal in the task description. Here the instruction names **only the algorithm**
(`Implement breadth-first search …`) and the function is called `func`. The sub-goal at the pivot (e.g. *make a queue* vs
*make a stack*) is never written down; the model has to derive it from the algorithm's name.

```
p_src: "Implement A"  + shared prefill  ->  sub-goal of A      (e.g. BFS -> queue)
p_dst: "Implement B"  + shared prefill  ->  sub-goal of B      (e.g. DFS -> stack)
```
**Intervention** = path patching (`src.diff.path_patch.intervention_stack`, the batch eval's code), I1 off:

* **I3, pivot-onwards KV transfer** (always on): the goal-250 heads read SRC's k/v over the task span **only from the last
  query on** (the pivot on the prefill pass, then each generated token), so every pre-pivot DST latent is untouched.
  Task span (`SPAN`): `"name+signature"` = the algorithm name in the instruction + the `def func(...):` line of the prefill
  (default) · `"name"` · `"instruction"` (the whole user turn). The name span is `\n`-padded to equal length (`pad`); the
  signature is token-identical in both prompts. Values ×`V_SCALE` (keys / attention pattern unchanged), OV boost `ALPHA` at -1.
* **I2, pre-commitment-swap residual transfer** (optional, `I2`): every layer's output residual over `<|im_end|> … pivot-1`
  is replaced with **SRC's** (the pairs have no third algorithm to act as p_res, so p_src is the donor — the Appendix J
  pre-commitment swap).

**Readouts** — GoalScope at the pivot (last prompt token) plus any prompt positions in `READ_AT`, on clean SRC / DST /
patched DST. Under I3 only the last query reads SRC's span, so patched readouts at earlier positions differ from clean DST
only through I2.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")   # check nvidia-smi first (shared machine)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import json, re
from contextlib import ExitStack
import torch
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
from src.diff.functional import (align_kv_spans, boost_head_ov, cache_src_proj, get_prompt, greedy,
                                 headwise_kv_patch, heads_per_layer, token_span)
from src.goals.goalscope import apply_goalscope
from src.tokens import prepare_input
from src.utils import env_utils

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
from rich import print as rich_print
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text

with open(os.path.join(os.path.dirname(env_utils.DEFAULT_RESULTS_DIR),
                       "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json")) as f:
    GOAL_HEADS_250 = [(l, h) for l, h in json.load(f)]
print(f"{len(GOAL_HEADS_250)} goal heads")

2026-10-19 11:17:56 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228
250 goal heads


## Knobs

In [2]:
#------------------------------------------------------------------------------------------------------
SPAN = "name"     # I3 task span: "name+signature" (algorithm name + `def func(...):` line) | "name" | "instruction"
V_SCALE = 4.0               # I3: x the transplanted SRC values (attention pattern unchanged); 1.0 = off
ALPHA = 1.0                 # I3: OV-write boost of PATCH_HEADS at -1 (pivot / each generated token); 1.0 = off
I2 = False                  # I2: pre-commitment swap — DST's <|im_end|> .. pivot-1 residuals (every layer) <- SRC's
ALIGN_KNOB = "pad"          # "pad" = newline-pad the shorter name span so the dest span is fully overwritten | "right"
PATCH_HEADS = GOAL_HEADS_250
READOUT_HEADS = GOAL_HEADS_250
MAX_NEW = 512                # greedy tokens per generation
RO_MAX_NEW = 64             # tokens per GoalScope verbalization
ORACLE = True               # grade generations + pivot readouts with the oracle (src.diff.eval.classify_concept)
ORACLE_MODEL = "claude-sonnet-4-6"
#------------------------------------------------------------------------------------------------------
from types import SimpleNamespace
import src.diff.path_patch as PP
PP_CTX = SimpleNamespace(mt=mt, group=P.GROUP, readout_heads=READOUT_HEADS, gsp=getattr(P, "gsp", None))

## Contrastive pairs — pick one

Each pair: two algorithms with the same signature and a shared code prefix that is natural for both; the pivot is the
last prefill token (the indentation token before the first diverging line — Qwen tokenizes `\n    x` as `\n` … `'   '`, `' x'`).
`kw` regexes give a quick verdict on the first line the model writes. `FLIP=True` swaps SRC and DST.

In [3]:
INSTRUCTION = "Implement {name} in {lang}. Name the function `func`."
LANG_NAME = {"python": "Python", "cpp": "C++"}

PAIRS = {
    "bfs_dfs": dict(
        a=dict(id="bfs", name="breadth-first search graph traversal", kw=r"queue|deque|popleft"),
        b=dict(id="dfs", name="iterative depth-first search graph traversal", kw=r"stack"),
        prefill="<code>\ndef func(graph, start):\n    visited = set()\n   ",
    ),
    "bsearch_leftmost": dict(
        a=dict(id="binary_search", name="binary search", kw=r"==\s*target"),
        b=dict(id="leftmost", name="binary search that returns the index of the leftmost occurrence of target",
               kw=r"<\s*target|result|ans|res\b"),
        prefill=("<code>\ndef func(arr, target):\n    lo, hi = 0, len(arr) - 1\n    while lo <= hi:\n"
                 "        mid = (lo + hi) // 2\n       "),
    ),
    "merge_quick": dict(
        a=dict(id="merge_sort", name="merge sort", kw=r"mid|merge|//\s*2"),
        b=dict(id="quick_sort", name="quick sort", kw=r"pivot"),
        prefill="<code>\ndef func(arr):\n    if len(arr) <= 1:\n        return arr\n   ",
    ),
    # ── added 2026-10-18; clean py first lines checked on Coder-32B (greedy): every side hits only its own kw ──
    "dijkstra_bfs": dict(
        a=dict(id="dijkstra", name="Dijkstra's shortest path algorithm from a start node", kw=r"heap|priority|\(\s*0\s*,"),
        b=dict(id="bfs_shortest", name="unweighted shortest path from a start node using breadth-first search",
               kw=r"deque|\[start\]"),
        prefill="<code>\ndef func(graph, start):\n    dist = {start: 0}\n   ",
    ),
    "prim_kruskal": dict(
        a=dict(id="prim", name="Prim's minimum spanning tree algorithm", kw=r"heap|adj|graph|visited|defaultdict"),
        b=dict(id="kruskal", name="Kruskal's minimum spanning tree algorithm", kw=r"sort|parent|find|rank"),
        prefill="<code>\ndef func(n, edges):\n    mst = []\n   ",
    ),
    "bubble_selection": dict(
        a=dict(id="bubble_sort", name="bubble sort", kw=r"n\s*-\s*i\s*-\s*1|\+\s*1\]"),
        b=dict(id="selection_sort", name="selection sort", kw=r"min"),
        prefill="<code>\ndef func(arr):\n    n = len(arr)\n    for i in range(n):\n       ",
    ),
    "fib_rec_iter": dict(
        a=dict(id="fib_recursive", name="the n-th Fibonacci number using naive recursion", kw=r"func\("),
        b=dict(id="fib_iterative", name="the n-th Fibonacci number iteratively with two running variables",
               kw=r"a,\s*b|prev|curr"),
        prefill="<code>\ndef func(n):\n    if n < 2:\n        return n\n   ",
    ),
    "power_fast_naive": dict(
        a=dict(id="fast_power", name="exponentiation by squaring", kw=r"while"),
        b=dict(id="naive_power", name="integer power by repeated multiplication in a loop", kw=r"for"),
        prefill="<code>\ndef func(base, exp):\n    result = 1\n   ",
    ),
    "twosum_hash_pointers": dict(
        a=dict(id="twosum_hash", name="two-sum that returns the indices of the two numbers, using a hash map",
               kw=r"seen|\{\}|dict|map"),
        b=dict(id="twosum_pointers",
               name="two-sum on a sorted array that returns the indices of the two numbers, using two pointers",
               kw=r"left|right|\blo\b|\bhi\b|\bi\s*,\s*j\b"),
        prefill="<code>\ndef func(nums, target):\n   ",
    ),
    "palindrome_reverse_pointers": dict(
        a=dict(id="pal_reverse", name="a palindrome check that compares the string with its reverse", kw=r"\[::-1\]"),
        b=dict(id="pal_pointers", name="a palindrome check that walks two pointers inward from both ends",
               kw=r"left|right|\blo\b|\bhi\b|\bi\s*,\s*j\b"),
        prefill="<code>\ndef func(s):\n   ",
    ),
    "toposort_kahn_dfs": dict(
        a=dict(id="kahn", name="topological sort using Kahn's algorithm", kw=r"in_?degree"),
        b=dict(id="topo_dfs", name="topological sort using depth-first search", kw=r"visit"),
        prefill="<code>\ndef func(graph):\n    result = []\n   ",
    ),
    "lcs_dp_memo": dict(
        a=dict(id="lcs_bottom_up", name="longest common subsequence length with bottom-up dynamic programming",
               kw=r"dp|table"),
        b=dict(id="lcs_top_down", name="longest common subsequence length with top-down memoized recursion",
               kw=r"memo|cache|def "),
        prefill="<code>\ndef func(a, b):\n    m, n = len(a), len(b)\n   ",
    ),
}

# C++ prefills — same shared prefix (natural for both algorithms), ending at the indentation token before the first
# diverging line. Not yet checked on the model (clean C++ generations): verify both sides diverge before trusting a pair.
PREFILL_CPP = {
    "bfs_dfs": "<code>\nstd::vector<int> func(std::vector<std::vector<int>>& graph, int start) {\n    std::vector<bool> visited(graph.size(), false);\n   ",
    "bsearch_leftmost": ("<code>\nint func(std::vector<int>& arr, int target) {\n    int lo = 0, hi = arr.size() - 1;\n"
                         "    while (lo <= hi) {\n        int mid = lo + (hi - lo) / 2;\n       "),
    "merge_quick": "<code>\nstd::vector<int> func(std::vector<int> arr) {\n    if (arr.size() <= 1) {\n        return arr;\n    }\n   ",
    "dijkstra_bfs": ("<code>\nstd::vector<int> func(std::vector<std::vector<int>>& graph, int start) {\n"
                     "    std::vector<int> dist(graph.size(), INT_MAX);\n    dist[start] = 0;\n   "),
    "prim_kruskal": "<code>\nint func(int n, std::vector<std::vector<int>>& edges) {\n    int total = 0;\n   ",
    "bubble_selection": "<code>\nvoid func(std::vector<int>& arr) {\n    int n = arr.size();\n    for (int i = 0; i < n - 1; i++) {\n       ",
    "fib_rec_iter": "<code>\nlong long func(int n) {\n    if (n < 2) {\n        return n;\n    }\n   ",
    "power_fast_naive": "<code>\nlong long func(long long base, int exp) {\n    long long result = 1;\n   ",
    "twosum_hash_pointers": "<code>\nstd::vector<int> func(std::vector<int>& nums, int target) {\n   ",
    "palindrome_reverse_pointers": "<code>\nbool func(const std::string& s) {\n   ",
    "toposort_kahn_dfs": "<code>\nstd::vector<int> func(std::unordered_map<int, std::vector<int>>& graph) {\n    std::vector<int> result;\n   ",
    "lcs_dp_memo": "<code>\nint func(const std::string& a, const std::string& b) {\n    int m = a.size(), n = b.size();\n   ",
}
# C++ keyword overrides for the quick first-line rule (the oracle is the real eval; these only fix idioms the
# Python regexes miss)
KW_CPP = {("dijkstra_bfs", "b"): r"deque|\[start\]|(?<!priority_)queue<",
          ("fib_rec_iter", "b"): r"a,\s*b|a\s*=\s*0|prev|curr",
          ("palindrome_reverse_pointers", "a"): r"\[::-1\]|rbegin|reverse"}
for k, pre in PREFILL_CPP.items():
    PAIRS[k]["prefill_cpp"] = pre
    for side in ("a", "b"):
        PAIRS[k][side]["kw_cpp"] = KW_CPP.get((k, side), PAIRS[k][side]["kw"])


def prefill_of(pair, lang):
    return pair["prefill"] if lang == "python" else pair["prefill_cpp"]


def kw_of(alg, lang):
    return alg["kw"] if lang == "python" else alg["kw_cpp"]


SRC_LANG, DST_LANG = "python", "cpp"   # <- "python" | "cpp" each; cross-language needs SPAN="name"|"instruction" and I2=False

PAIR, FLIP = "merge_quick", False      # <- choose
# GoalScope positions besides the pivot (always included). Inside the prompt only.
#   int < 0  -> offset from the end of the prompt (-1 = pivot)      int >= 0 -> absolute index in the DST tokenization
#   str      -> last token in the prompt whose text (stripped) equals it, e.g. " visited" (skipped if absent)
READ_AT = [-2, " visited", " func"]

## Build SRC / DST prompts — task span (yellow), transplanted k/v window (orange), pivot (salmon)

In [4]:
def question(alg, lang="python"):
    return INSTRUCTION.format(name=alg["name"], lang=LANG_NAME[lang])


def span_text(alg, lang="python"):
    return question(alg, lang) if SPAN == "instruction" else alg["name"]


def signature_line(prefill):
    return next(ln for ln in prefill.split("\n") if "func(" in ln)


def build(pair_key, flip=False, src_lang=None, dst_lang=None):
    """Prompts + the I3 window (src_w / dst_w, paired 1:1) + the I2 window (seg on DST, res_seg on SRC), as the
    `g` dict PP.intervention_stack expects (I2 donor = SRC: res_tok = src_tok). SRC / DST are written in
    `src_lang` / `dst_lang` (default SRC_LANG / DST_LANG). Across languages only the instruction-side window works:
    the signature line and the progress prefill differ in tokens, so "name+signature" and I2 need one language."""
    pair = PAIRS[pair_key]
    sl, dl = src_lang or SRC_LANG, dst_lang or DST_LANG
    xlang = sl != dl
    if xlang and SPAN == "name+signature":
        raise ValueError(f"SPAN='name+signature' needs SRC and DST in one language ({sl} -> {dl}); use 'name'")
    if xlang and I2:
        raise ValueError(f"I2 swaps token-identical progress latents; {sl} -> {dl} prefills differ — set I2=False")
    src, dst = (pair["b"], pair["a"]) if flip else (pair["a"], pair["b"])
    src_prompt = get_prompt(question(src, sl), prefill_of(pair, sl))
    dst_prompt = get_prompt(question(dst, dl), prefill_of(pair, dl))
    s0, t0 = token_span(src_prompt, span_text(src, sl)), token_span(dst_prompt, span_text(dst, dl))
    src_tok, dst_tok, src_pos, dst_pos, src_span, dst_span = align_kv_spans(
        mt, src_prompt, dst_prompt, s0, t0, mode=ALIGN_KNOB)
    src_w, dst_w = list(src_pos), list(dst_pos)
    s_ids, d_ids = src_tok["input_ids"][0].tolist(), dst_tok["input_ids"][0].tolist()
    L_s, L_d = len(s_ids), len(d_ids)
    if SPAN == "name+signature":   # + the `def func(...):` line (token-identical; located in DST, aligned from the end)
        sig = signature_line(prefill_of(pair, dl))
        lo, hi = token_span(dst_prompt, sig)
        shift = L_d - prepare_input(prompts=dst_prompt, tokenizer=mt)["input_ids"].shape[1]   # pad tokens before it
        sig_d = list(range(lo + shift, hi + shift))
        sig_s = [q - L_d + L_s for q in sig_d]
        assert [s_ids[q] for q in sig_s] == [d_ids[q] for q in sig_d], "signature tokens differ"
        src_w, dst_w = src_w + sig_s, dst_w + sig_d
    # I2 window: last <|im_end|> .. pivot-1, token-identical in both prompts (aligned from the end) — one language only
    seg, res_seg = [], []
    if not xlang:
        im_end = mt.tokenizer.convert_tokens_to_ids("<|im_end|>")
        seg = list(range(max(i for i, t in enumerate(d_ids) if t == im_end), L_d - 1))
        res_seg = [q - L_d + L_s for q in seg]
        assert [s_ids[q] for q in res_seg] == [d_ids[q] for q in seg], "SRC/DST progress tokens differ"
    return dict(pair=pair_key, flip=flip, src=src, dst=dst, src_lang=sl, dst_lang=dl, src_tok=src_tok, dst_tok=dst_tok,
                src_pos=src_w, dst_pos=dst_w, src_span=src_span, dst_span=dst_span,
                src_w=src_w, dst_w=dst_w, seg=seg, res_tok=src_tok, res_seg=res_seg)


def show_tokens(tok, title, task_span, kv_pos):
    ids = tok["input_ids"][0].tolist()
    task, kv, piv = set(range(*task_span)), set(kv_pos), len(ids) - 1
    t = Text()
    for i, tid in enumerate(ids):
        s = mt.tokenizer.decode([tid])
        style = ("black on salmon1" if i == piv else "black on dark_orange" if i in kv
                 else "bright_yellow" if i in task else "")
        t.append(s, style=style)
    rich_print(Panel(t, title=f"[bold]{title}[/bold]  L={len(ids)}  pivot={mt.tokenizer.decode([ids[-1]])!r}",
                     title_align="left", border_style="grey42"))


def resolve_positions(tok, read_at):
    """READ_AT -> sorted absolute positions in `tok` (pivot always included)."""
    ids = tok["input_ids"][0].tolist()
    n, out = len(ids), {len(ids) - 1}
    for p in read_at:
        if isinstance(p, int):
            out.add(n + p if p < 0 else p)
        else:
            hits = [i for i, t in enumerate(ids) if mt.tokenizer.decode([t]).strip() == p.strip()]
            if not hits:
                print(f"READ_AT: token {p!r} not in this prompt — skipped")
                continue
            out.add(hits[-1])
    assert all(0 <= q < n for q in out), f"positions out of range: {sorted(out)} (L={n})"
    return sorted(out)


B = build(PAIR, FLIP)
for name, tok, span, pos in [("SRC", B["src_tok"], B["src_span"], B["src_pos"]),
                             ("DST", B["dst_tok"], B["dst_span"], B["dst_pos"])]:
    alg = B["src"] if name == "SRC" else B["dst"]
    lang = B["src_lang"] if name == "SRC" else B["dst_lang"]
    show_tokens(tok, f"{name} · {alg['id']} · {lang} · span={SPAN} · I3 window (orange) {len(pos)} tok", span, pos)
print(f"SRC = {B['src']['id']} ({B['src_lang']})   DST = {B['dst']['id']} ({B['dst_lang']})   (I3: SRC span k/v -> DST, last-onwards"
      f"{' · I2: DST progress <- SRC ' + str((B['seg'][0], B['seg'][-1])) if I2 else ''})")

╭─ SRC · merge_sort · python · span=name · I3 window (orange) 2 tok  L=78  pivot='   ' ───────────────────────────╮
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Implement merge sort in Python. Name the function `func`.<|im_end|>                                             │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def func(arr):                                                                                                  │
│     if len(arr) <= 1:                                                                                           │
│         return arr                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · quick_sort · cpp · span=name · I3 window (orange) 2 tok  L=92  pivot='   ' ──────────────────────────────╮
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Implement quick sort in C++. Name the function `func`.<|im_end|>                                                │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ std::vector<int> func(std::vector<int> arr) {                                                                   │
│     if (arr.size() <= 1) {                                                                                      │
│         return arr;                                                                                             │
│     }                                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

SRC = merge_sort (python)   DST = quick_sort (cpp)   (I3: SRC span k/v -> DST, last-onwards)


### Token picker — the prompt tail with absolute index and offset from end (for `READ_AT`)

In [5]:
def token_picker(tok, tail=40):
    ids = tok["input_ids"][0].tolist()
    n = len(ids)
    t = Table(show_header=True, header_style="bold", box=None, pad_edge=False)
    for c in ("abs", "from end", "token"):
        t.add_column(c, justify="right" if c != "token" else "left")
    for i in range(max(0, n - tail), n):
        t.add_row(str(i), str(i - n), repr(mt.tokenizer.decode([ids[i]])))
    rich_print(t)


token_picker(B["dst_tok"])
print("READ_AT ->", resolve_positions(B["dst_tok"], READ_AT))

abs  from end  token         
 52       -40  '<|im_end|>'  
 53       -39  '\n'          
 54       -38  '<|im_start|>'
 55       -37  'assistant'   
 56       -36  '\n'          
 57       -35  '<'           
 58       -34  'code'        
 59       -33  '>\n'         
 60       -32  'std'         
 61       -31  '::'          
 62       -30  'vector'      
 63       -29  '<int'        
 64       -28  '>'           
 65       -27  ' func'       
 66       -26  '(std'        
 67       -25  '::'          
 68       -24  'vector'      
 69       -23  '<int'        
 70       -22  '>'           
 71       -21  ' arr'        
 72       -20  ')'           
 73       -19  ' {\n'        
 74       -18  '   '         
 75       -17  ' if'         
 76       -16  ' ('          
 77       -15  'arr'         
 78       -14  '.size'       
 79       -13  '()'          
 80       -12  ' <='         
 81       -11  ' '           
 82       -10  '1'           
 83        -9  ')'           
 84        -8  ' {\n'        
 85        -7  '       '     
 86        -6  ' return'     
 87        -5  ' arr'        
 88        -4  ';\n'         
 89        -3  '   '         
 90        -2  ' }\n'        
 91        -1  '   '

READ_AT: token ' visited' not in this prompt — skipped
READ_AT -> [65, 90, 91]


## Intervention — generations + GoalScope readouts

In [6]:
def verdict(gen, b, lang="python"):
    first = next((ln for ln in gen.split("\n") if ln.strip()), "")
    hs, hd = bool(re.search(kw_of(b["src"], lang), first)), bool(re.search(kw_of(b["dst"], lang), first))
    return "SRC" if hs and not hd else "DST" if hd and not hs else "both" if hs else "neither"


from src.diff.eval import classify_concept

ORACLE_TASK = ("Write a function `func` (in Python or C++) that implements a named algorithm. The function starts "
               "with a prefilled prefix; judge the algorithm the function implements")
O2V = {"src": "SRC", "dst": "DST", "both": "both", "neither": "neither", "parse_error": "parse_error"}


def function_text(b, gen, lang="python"):
    """The whole function as written: the prefill in `lang` (without <code>) + the continuation up to </code>."""
    return prefill_of(PAIRS[b["pair"]], lang).replace("<code>\n", "", 1) + gen.split("</code>")[0]


def oracle_code(b, gen, lang="python"):
    r = classify_concept(function_text(b, gen, lang), f"implements {b['src']['name']}", f"implements {b['dst']['name']}",
                         ORACLE_TASK, kind=f"a {LANG_NAME[lang]} function: a prefilled prefix followed by the model's continuation. "
                         "Judge which ALGORITHM the function as a whole implements (BOTH = a mix of the two algorithms)",
                         model_name=ORACLE_MODEL)
    return dict(r, verdict=O2V[r["label"]])


def oracle_readout(b, text):
    r = classify_concept(text, f"implements {b['src']['name']}", f"implements {b['dst']['name']}", ORACLE_TASK,
                         kind="a short natural-language description of what the model is doing at its current "
                              "position in the code (a verbalized readout of its internal state)",
                         model_name=ORACLE_MODEL)
    return dict(r, verdict=O2V[r["label"]])


def run(b, read_at=READ_AT):
    src_tok, dst_tok = b["src_tok"], b["dst_tok"]
    # I3 (last-onwards) [+ I2] — the path-patching batch eval's intervention stack (fresh single-use stack per forward)
    patched = PP.intervention_stack(b, task_heads=PATCH_HEADS, goal_heads=PATCH_HEADS, i1=False, i2=I2, i3=True,
                                    i3_query="last", v_scale=V_SCALE, alpha=ALPHA, donor="res", ctx=PP_CTX)

    def patched_iface(mt_, inputs, patches=None, model_kwargs=None):
        with patched():
            return mt_._model(**dict(inputs), **dict(model_kwargs or {}))

    # generations: I3 reads SRC's span at the last query of every forward = pivot on prefill, then each generated token
    gens = dict(src=greedy(src_tok, max_new=MAX_NEW), dst=greedy(dst_tok, max_new=MAX_NEW))
    with patched():
        gens["patched"] = greedy(dst_tok, max_new=MAX_NEW)

    # readout positions: resolved on DST, mapped to SRC by offset from the end (identical under pad). Across languages
    # the prefills differ, so SRC is read out at its pivot only (other rows show "—" for clean SRC).
    n_s, n_d = src_tok["input_ids"].shape[1], dst_tok["input_ids"].shape[1]
    pos_d = resolve_positions(dst_tok, read_at)
    xlang = b["src_lang"] != b["dst_lang"]
    pos_s = [(n_s - 1 if pd == n_d - 1 else None) if xlang else pd - n_d + n_s for pd in pos_d]
    for ps, pd in zip(pos_s, pos_d):
        if ps is None:
            continue
        ts, td = (mt.tokenizer.decode([t["input_ids"][0, q]]) for t, q in ((src_tok, ps), (dst_tok, pd)))
        if ts != td:
            print(f"warning: SRC pos {ps} {ts!r} != DST pos {pd} {td!r} (position is inside the differing span?)")

    def ro(tok, pos, iface=None):
        res = apply_goalscope(prompt=tok, mt=mt, selected_heads=READOUT_HEADS, strategy="additive_norm",
                              alpha=0.1, steer_layer_range=(35, 46), batch_size=len(pos),
                              max_new_tokens=RO_MAX_NEW, goalscope_prompt=getattr(P, "gsp", None),
                              token_positions=pos, include_noncode=False, show=False, patch_interface=iface)
        return {r["pos"]: r["patched_gen"] for r in res}

    ros = dict(src=ro(src_tok, [q for q in pos_s if q is not None]), dst=ro(dst_tok, pos_d),
               patched=ro(dst_tok, pos_d, iface=patched_iface))
    rows = [dict(pos=pd, from_end=pd - n_d, token=mt.tokenizer.decode([dst_tok["input_ids"][0, pd]]),
                 src=ros["src"][ps] if ps is not None else "—", dst=ros["dst"][pd], patched=ros["patched"][pd])
            for ps, pd in zip(pos_s, pos_d)]
    lang = dict(src=b["src_lang"], dst=b["dst_lang"], patched=b["dst_lang"])   # language each generation is written in
    out = dict(gens=gens, verdicts={k: verdict(g, b, lang[k]) for k, g in gens.items()}, rows=rows)
    if ORACLE:   # oracle verdicts: the whole function per run, and the pivot readout per run
        piv = next(r for r in rows if r["from_end"] == -1)
        out["oracle"] = {k: oracle_code(b, g, lang[k]) for k, g in gens.items()}
        out["oracle_ro"] = {k: oracle_readout(b, piv[k]) for k in ("src", "dst", "patched")}
    return out


def show(b, out):
    head = (f"{b['pair']}{' (flipped)' if b['flip'] else ''} · SRC={b['src']['id']} ({b['src_lang']}) -> "
            f"DST={b['dst']['id']} ({b['dst_lang']}) · "
            f"I3 last-onwards · span={SPAN} · V×{V_SCALE:g} · α={ALPHA} · {ALIGN_KNOB} · I2={'on (SRC donor)' if I2 else 'off'}")
    for k, col in (("src", "red1"), ("dst", "sea_green3"), ("patched", "salmon1")):
        o = (f"  oracle={out['oracle'][k]['verdict']} · readout={out['oracle_ro'][k]['verdict']}"
             if "oracle" in out else "")
        rich_print(Panel(Text(out["gens"][k]), title=f"[{col}]{k}[/{col}]  kw={out['verdicts'][k]}{o}",
                         title_align="left", border_style="grey42"))
    t = Table(title=head, show_lines=True)
    t.add_column("pos", justify="right"); t.add_column("from end", justify="right"); t.add_column("token")
    for k, col in (("src", "red1"), ("dst", "sea_green3"), ("patched", "salmon1")):
        t.add_column(Text(f"{'clean SRC' if k == 'src' else 'DST' if k == 'dst' else 'patched DST'}", style=col), ratio=1)
    for r in out["rows"]:
        tok = Text(repr(r["token"]), style="bold salmon1" if r["from_end"] == -1 else "")
        t.add_row(str(r["pos"]), str(r["from_end"]), tok, *(Text(r[k].strip()) for k in ("src", "dst", "patched")))
    Console(width=200).print(t)


OUT = run(B)
show(B, OUT)

READ_AT: token ' visited' not in this prompt — skipped


GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

╭─ src  kw=SRC  oracle=SRC · readout=SRC ─────────────────────────────────────────────────────────────────────────╮
│  mid = len(arr) // 2                                                                                            │
│     left_half = func(arr[:mid])                                                                                 │
│     right_half = func(arr[mid:])                                                                                │
│     return merge(left_half, right_half)                                                                         │
│                                                                                                                 │
│ def merge(left, right):                                                                                         │
│     sorted_array = []                                                                                           │
│     left_index, right_index = 0, 0                                                                              │
│     while left_index < len(left) and right_index < len(right):                                                  │
│         if left[left_index] < right[right_index]:                                                               │
│             sorted_array.append(left[left_index])                                                               │
│             left_index += 1                                                                                     │
│         else:                                                                                                   │
│             sorted_array.append(right[right_index])                                                             │
│             right_index += 1                                                                                    │
│     sorted_array.extend(left[left_index:])                                                                      │
│     sorted_array.extend(right[right_index:])                                                                    │
│     return sorted_array                                                                                         │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ dst  kw=DST  oracle=DST · readout=DST ─────────────────────────────────────────────────────────────────────────╮
│  int pivot = arr[arr.size() / 2];                                                                               │
│     std::vector<int> left, right, equal;                                                                        │
│     for (int num : arr) {                                                                                       │
│         if (num < pivot) {                                                                                      │
│             left.push_back(num);                                                                                │
│         } else if (num > pivot) {                                                                               │
│             right.push_back(num);                                                                               │
│         } else {                                                                                                │
│             equal.push_back(num);                                                                               │
│         }                                                                                                       │
│     }                                                                                                           │
│     left = func(left);                                                                                          │
│     right = func(right);                                                                                        │
│     left.insert(left.end(), equal.begin(), equal.end());                                                        │
│     left.insert(left.end(), right.begin(), right.end());                                                        │
│     return left;                                                                                                │
│ }                                                                                                               │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ patched  kw=SRC  oracle=SRC · readout=SRC ─────────────────────────────────────────────────────────────────────╮
│  int mid = arr.size() / 2;                                                                                      │
│     std::vector<int> left(arr.begin(), arr.begin() + mid);                                                      │
│     std::vector<int> right(arr.begin() + mid, arr.end());                                                       │
│     left = func(left);                                                                                          │
│     right = func(right);                                                                                        │
│     arr.clear();                                                                                                │
│     while (!left.empty() || !right.empty()) {                                                                   │
│         if (left.empty()) {                                                                                     │
│             arr.push_back(right.front());                                                                       │
│             right.erase(right.begin());                                                                         │
│         } else if (right.empty()) {                                                                             │
│             arr.push_back(left.front());                                                                        │
│             left.erase(left.begin());                                                                           │
│         } else if (left.front() < right.front()) {                                                              │
│             arr.push_back(left.front());                                                                        │
│             left.erase(left.begin());                                                                           │
│         } else {                                                                                                │
│             arr.push_back(right.front());                                                                       │
│             right.erase(right.begin());                                                                         │
│         }                                                                                                       │
│     }                                                                                                           │
│     return arr;                                                                                                 │
│ }                                                                                                               │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                        merge_quick · SRC=merge_sort (python) -> DST=quick_sort (cpp) · I3 last-onwards · span=name · V×4 · α=1.0 · pad · I2=off                                        
┏━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pos ┃ from end ┃ token   ┃ clean SRC                                              ┃ DST                                                     ┃ patched DST                                            ┃
┡━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  65 │      -27 │ ' func' │ —                                                      │ I'm not actually executing any code here; I'm providing │ I'm not actually executing any code here; I'm          │
│     │          │         │                                                        │ a function to sort an array using quicksort. Here's a   │ providing a function to sort an array using quicksort. │
│     │          │         │                                                        │ brief implementation:                                   │ Here's a brief implementation:                         │
│     │          │         │                                                        │                                                         │                                                        │
│     │          │         │                                                        │ ```cpp                                                  │ ```cpp                                                 │
│     │          │         │                                                        │ void quicksort(int arr[], int low, int high) {          │ void quicksort(int arr[], int low, int high) {         │
│     │          │         │                                                        │     if (low < high) {                                   │     if (low < high) {                                  │
│     │          │         │                                                        │         int partitionIndex = partition(arr, low, high); │         int partitionIndex = partition(arr, low,       │
│     │          │         │                                                        │                                                         │ high);                                                 │
├─────┼──────────┼─────────┼────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│  90 │       -2 │ ' }\n'  │ —                                                      │ Sure, please provide the specific line or snippet of    │ Sure, please provide the specific line or snippet of   │
│     │          │         │                                                        │ code you want me to analyze, and I'll help with that.   │ code you want me to analyze, and I'll help with that.  │
├─────┼──────────┼─────────┼────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│  91 │       -1 │ '   '   │ I'm not executing any code; I'm just preparing to      │ I'm not executing any code directly; I'm using a        │ I'm not executing any code directly; I'm here to help  │
│     │          │         │ divide the task into smaller parts and merge them back │ divide-and-conquer approach to sort it. Here's a brief  │ you by dividing your request into smaller parts,       │
│     │          │         │ together. No action is taken at a single point; the    │ explanation:                                            │ sorting those parts, and then merging

# scaled up eval

Sweeps every `(SRC, DST)` language pair in `LANG_SWEEP` (py/C++), every pair, both directions.


In [7]:
# ── sweep: every language pair × every pair × both directions — generations + readouts inline, scores per config ──
LANG_SWEEP = [("python", "python"), ("python", "cpp"), ("cpp", "python"), ("cpp", "cpp")]   # (SRC, DST) languages
CFG = f"span={SPAN} · V×{V_SCALE:g} · α={ALPHA} · I2={'on' if I2 else 'off'}"
SWEEP = globals().get("SWEEP", {})          # {config: [row, ...]} — re-running with other knobs adds a config
SWEEP[CFG] = []
for sl, dl in LANG_SWEEP:
    if sl != dl and (SPAN == "name+signature" or I2):   # cross-language needs SPAN "name"/"instruction" and I2 off
        rich_print(f"[yellow]skip {sl}→{dl}: span={SPAN}, I2={I2} needs one language[/yellow]")
        continue
    for key, flip in [(k, f) for k in list(PAIRS.keys()) for f in (False, True)]:
        b = build(key, flip, src_lang=sl, dst_lang=dl)
        out = run(b)
        show(b, out)
        piv = next(r for r in out["rows"] if r["from_end"] == -1)
        assert "oracle" in out, "set ORACLE = True — the sweep is scored with the oracle"
        v = {k: o["verdict"] for k, o in out["oracle"].items()}         # oracle: which algorithm the function implements
        vr = {k: o["verdict"] for k, o in out["oracle_ro"].items()}     # oracle: which algorithm the pivot readout names
        kw = out["verdicts"]                                             # keyword rule on the first line (comparison)
        SWEEP[CFG].append(dict(pair=key, flip=flip, src=b["src"]["id"], dst=b["dst"]["id"], src_lang=sl, dst_lang=dl,
                               v_src=v["src"], v_dst=v["dst"], v_patched=v["patched"],
                               valid=(v["src"] == "SRC" and v["dst"] == "DST"), shifted=(v["patched"] == "SRC"),
                               r_src=vr["src"], r_dst=vr["dst"], r_patched=vr["patched"],
                               ro_valid=(vr["src"] == "SRC" and vr["dst"] == "DST"), ro_shifted=(vr["patched"] == "SRC"),
                               kw_src=kw["src"], kw_dst=kw["dst"], kw_patched=kw["patched"],
                               why_patched=out["oracle"]["patched"]["why"],
                               ro_src=piv["src"], ro_dst=piv["dst"], ro_patched=piv["patched"],
                               gen_patched=out["gens"]["patched"]))
        r = SWEEP[CFG][-1]
        rich_print(f"[bold]{sl}→{dl} · {key}{' (flipped)' if flip else ''}[/bold] {r['src']} → {r['dst']} · clean {r['v_src']}/{r['v_dst']} "
                   f"({'valid' if r['valid'] else 'INVALID'}) · patched [bold]{r['v_patched']}[/bold] · "
                   f"readout {r['r_src']}/{r['r_dst']} → [bold]{r['r_patched']}[/bold]\n")

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

╭─ src  kw=SRC  oracle=SRC · readout=SRC ─────────────────────────────────────────────────────────────────────────╮
│  queue = [start]                                                                                                │
│     while queue:                                                                                                │
│         vertex = queue.pop(0)                                                                                   │
│         if vertex not in visited:                                                                               │
│             visited.add(vertex)                                                                                 │
│             queue.extend([neighbor for neighbor in graph[vertex] if neighbor not in visited])                   │
│     return visited                                                                                              │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ dst  kw=DST  oracle=DST · readout=DST ─────────────────────────────────────────────────────────────────────────╮
│  stack = [start]                                                                                                │
│     while stack:                                                                                                │
│         vertex = stack.pop()                                                                                    │
│         if vertex not in visited:                                                                               │
│             visited.add(vertex)                                                                                 │
│             stack.extend([neighbor for neighbor in graph[vertex] if neighbor not in visited])                   │
│     return visited                                                                                              │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ patched  kw=SRC  oracle=SRC · readout=neither ─────────────────────────────────────────────────────────────────╮
│  queue = [start]                                                                                                │
│     while queue:                                                                                                │
│         vertex = queue.pop(0)                                                                                   │
│         if vertex not in visited:                                                                               │
│             visited.add(vertex)                                                                                 │
│             queue.extend([neighbor for neighbor in graph[vertex] if neighbor not in visited])                   │
│     return visited                                                                                              │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (846 outputs left out)

╭─ dst  kw=DST  oracle=DST · readout=DST ─────────────────────────────────────────────────────────────────────────╮
│  std::vector<std::vector<int>> dp(m + 1, std::vector<int>(n + 1, 0));                                           │
│     for (int i = 1; i <= m; ++i) {                                                                              │
│         for (int j = 1; j <= n; ++j) {                                                                          │
│             if (a[i - 1] == b[j - 1]) {                                                                         │
│                 dp[i][j] = dp[i - 1][j - 1] + 1;                                                                │
│             } else {                                                                                            │
│                 dp[i][j] = std::max(dp[i - 1][j], dp[i][j - 1]);                                                │
│             }                                                                                                   │
│         }                                                                                                       │
│     }                                                                                                           │
│     return dp[m][n];                                                                                            │
│ }                                                                                                               │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ patched  kw=SRC  oracle=SRC · readout=SRC ─────────────────────────────────────────────────────────────────────╮
│  std::vector<std::vector<int>> memo(m + 1, std::vector<int>(n + 1, -1));                                        │
│                                                                                                                 │
│     std::function<int(int, int)> lcs = [&](int i, int j) {                                                      │
│         if (i == 0 || j == 0) return 0;                                                                         │
│         if (memo[i][j] != -1) return memo[i][j];                                                                │
│         if (a[i - 1] == b[j - 1]) {                                                                             │
│             memo[i][j] = 1 + lcs(i - 1, j - 1);                                                                 │
│         } else {                                                                                                │
│             memo[i][j] = std::max(lcs(i - 1, j), lcs(i, j - 1));                                                │
│         }                                                                                                       │
│         return memo[i][j];                                                                                      │
│     };                                                                                                          │
│                                                                                                                 │
│     return lcs(m, n);                                                                                           │
│ }                                                                                                               │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                  lcs_dp_memo (flipped) · SRC=lcs_top_down (cpp) -> DST=lcs_bottom_up (cpp) · I3 last-onwards · span=name · V×4 · α=1.0 · pad · I2=off                                  
┏━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pos ┃ from end ┃ token   ┃ clean SRC                                              ┃ DST                                                     ┃ patched DST                                            ┃
┡━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  70 │      -29 │ ' func' │ I'm not executing any specific code here, but I can    │ I'm calculating the longest common subsequence (LCS) of │ I'm calculating the longest common subsequence (LCS)   │
│     │          │         │ help with that using memoization and recursion. Here's │ two strings using dynamic programming. Here's a brief   │ of two strings using dynamic programming. Here's a     │
│     │          │         │ a brief recursive function with memoization to find    │ implementation:                                         │ brief implementation:                                  │
│     │          │         │ the longest common subsequence (LCS) of two strings:   │                                                         │                                                        │
│     │          │         │                                                        │ ```cpp                                                  │ ```cpp                                                 │
│     │          │         │ ```cpp                                                 │ int lcs(const string& X, const string& Y) {             │ int lcs(const string& X, const string& Y) {            │
│     │          │         │ #include <vector>                                      │     int m = X.size(), n = Y.size();                     │     int m = X.size(), n = Y.size();                    │
│     │          │         │ using namespace std;                                   │     vector<vector<int>> dp(m + 1                        │     vector<vector<int>> dp(m + 1                       │
│     │          │         │                                                        │                                                         │                                                        │
│     │          │         │ int lcsHelper(const string& s                          │                                                         │                                                        │
├─────┼──────────┼─────────┼────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│  97 │       -2 │ '();\n' │ I'm checking if a value is already computed and stored │ I'm processing and comparing values stored in a 2D      │ I'm processing and comparing values stored in a 2D     │
│     │          │         │ in a memoization table to avoid redundant calculations │ array `dp` using dynamic programming to solve a         │ array `dp` using dynamic programming to solve a        │
│     │          │         │ in a recursive function.                               │ problem, typically a longest common subsequence         │ problem, typically a longest common subsequence        │
│     │          │         │                                                        │ problem.                                                │ problem.                                               │
├─────┼──────────┼─────────┼────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────┼──────────────────────────────────────

cpp→cpp · lcs_dp_memo (flipped) lcs_top_down → lcs_bottom_up · clean SRC/DST (valid) · patched SRC · readout 
SRC/DST → SRC

## Success rates

Scored by the **oracle** (`classify_concept`, `ORACLE_MODEL`): for each run it reads the whole function (shared prefill +
continuation) and says whether it implements SRC's algorithm, DST's, both (a mix), or neither; the pivot GoalScope readouts
are classified the same way.
* **causal-shift** = patched DST implements SRC's algorithm (`both` is a miss) · **valid** = clean SRC → SRC and clean DST → DST.
* **readout-shift** = the patched pivot readout names SRC's algorithm · **readout valid** = clean SRC / DST readouts → SRC / DST.
* **kw agreement** = how often the old first-line keyword rule agrees with the oracle on the patched run.

In [10]:
import collections

def rate(rows, f):
    return (sum(f(r) for r in rows) / len(rows)) if rows else float("nan")

for cfg, rows in SWEEP.items():
    valid = [r for r in rows if r["valid"]]
    ro_valid = [r for r in rows if r["ro_valid"]]
    t = Table(title=f"{cfg} · n={len(rows)} (valid {len(valid)})", show_lines=False)
    for c in ("langs", "pair", "direction", "clean SRC / DST", "patched", "readout SRC / DST → patched", "kw patched"):
        t.add_column(c)
    for r in rows:
        col = "green" if r["shifted"] else "red" if r["valid"] else "grey50"
        rcol = "green" if r["ro_shifted"] else "red" if r["ro_valid"] else "grey50"
        t.add_row(f"{r.get('src_lang', 'python')}→{r.get('dst_lang', 'python')}", r["pair"], f"{r['src']} → {r['dst']}", f"{r['v_src']} / {r['v_dst']}", Text(r["v_patched"], style=col),
                  Text(f"{r['r_src']} / {r['r_dst']} → {r['r_patched']}", style=rcol), r["kw_patched"])
    Console(width=180).print(t)
    rich_print(f"  causal-shift : all [bold]{rate(rows, lambda r: r['shifted']):.2f}[/bold] ({sum(r['shifted'] for r in rows)}/{len(rows)}) · "
               f"valid-only [bold]{rate(valid, lambda r: r['shifted']):.2f}[/bold] ({sum(r['shifted'] for r in valid)}/{len(valid)}) · "
               f"valid rate {len(valid) / len(rows):.2f}")
    rich_print(f"  readout-shift: all [bold]{rate(rows, lambda r: r['ro_shifted']):.2f}[/bold] · "
               f"readout-valid-only [bold]{rate(ro_valid, lambda r: r['ro_shifted']):.2f}[/bold] ({sum(r['ro_shifted'] for r in ro_valid)}/{len(ro_valid)}) · "
               f"readout valid rate {len(ro_valid) / len(rows):.2f}")
    rich_print(f"  patched verdicts: {dict(collections.Counter(r['v_patched'] for r in rows))} · "
               f"patched readouts: {dict(collections.Counter(r['r_patched'] for r in rows))} · "
               f"kw agreement {rate(rows, lambda r: r['kw_patched'] == r['v_patched']):.2f}")
    by_lang = {lp: [r for r in rows if (r.get("src_lang", "python"), r.get("dst_lang", "python")) == lp]
               for lp in dict.fromkeys((r.get("src_lang", "python"), r.get("dst_lang", "python")) for r in rows)}
    rich_print("  by language (causal · valid-only · readout): " + " · ".join(
        f"{s}→{d} {rate(rs, lambda r: r['shifted']):.2f} · {rate([r for r in rs if r['valid']], lambda r: r['shifted']):.2f} · "
        f"{rate(rs, lambda r: r['ro_shifted']):.2f} (n={len(rs)})" for (s, d), rs in by_lang.items()))
    by_flip = {f: [r for r in rows if r["flip"] == f] for f in (False, True)}
    rich_print("  by direction: " + " · ".join(f"{'flipped' if f else 'a→b'} {rate(rs, lambda r: r['shifted']):.2f}"
                                             for f, rs in by_flip.items()))
    by_pair = {k: [r for r in rows if r["pair"] == k] for k in dict.fromkeys(r["pair"] for r in rows)}
    rich_print("  by pair (causal): " + " · ".join(f"{k} {sum(r['shifted'] for r in rs)}/{len(rs)}" for k, rs in by_pair.items()) + "\n")

                                                  span=name · V×4 · α=1.0 · I2=off · n=96 (valid 96)                                                  
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ langs         ┃ pair                        ┃ direction                     ┃ clean SRC / DST ┃ patched ┃ readout SRC / DST → patched ┃ kw patched ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ python→python │ bfs_dfs                     │ bfs → dfs                     │ SRC / DST       │ SRC     │ SRC / DST → neither         │ SRC        │
│ python→python │ bfs_dfs                     │ dfs → bfs                     │ SRC / DST       │ SRC     │ SRC / DST → SRC             │ SRC        │
│ python→python │ bsearch_leftmost            │ binary_search → leftmost      │ SRC / DST       │ DST     │ SRC / DST → SRC             │ DST        │
│ python→python │ bsearch_leftmost            │ leftmost → binary_search      │ SRC / DST       │ SRC     │ SRC / DST → DST             │ DST        │
│ python→python │ merge_quick                 │ merge_sort → quick_sort       │ SRC / DST       │ SRC     │ SRC / DST → SRC             │ SRC        │
│ python→python │ merge_quick                 │ quick_sort → merge_sort       │ SRC / DST       │ SRC     │ SRC / DST → SRC             │ both       │
│ python→python │ dijkstra_bfs                │ dijkstra → bfs_shortest       │ SRC / DST       │ SRC     │ SRC / both → SRC            │ SRC        │
│ python→python │ dijkstra_bfs                │ bfs_shortest → dijkstra       │ SRC / DST       │ SRC     │ SRC / DST → neither         │ SRC        │
│ python→python │ prim_kruskal                │ prim → kruskal                │ SRC / DST       │ SRC     │ neither / DST → SRC         │ DST        │
│ python→python │ prim_kruskal                │ kruskal → prim                │ SRC / DST       │ SRC     │ SRC / neither → neither     │ SRC        │
│ python→python │ bubble_selection            │ bubble_sort → selection_sort  │ SRC / DST       │ SRC     │ SRC / DST → SRC             │ SRC        │
│ python→python │ bubble_selection            │ selection_sort → bubble_sort  │ SRC / DST       │ SRC     │ SRC / DST → SRC             │ SRC        │
│ python→python │ fib_rec_iter                │ fib_recursive → fib_iterative │ SRC / DST       │ DST     │ SRC / DST → SRC             │ DST        │
│ python→python │ fib_rec_iter                │ fib_iterative → fib_recursive │ SRC / DST       │ SRC     │ SRC / DST → SRC             │ SRC        │
│ python→python │ power_fast_naive            │ fast_power → naive_power      │ SRC / DST       │ DST     │ SRC / DST → SRC             │ DST        │
│ python→python │ power_fast_naive            │ naive_power → fast_power      │ SRC / DST       │ DST     │ SRC / DST → SRC             │ DST        │
│ python→python │ twosum_hash_pointers        │ twosum_hash → twosum_pointers │ SRC / DST       │ SRC     │ SRC / DST → SRC             │ SRC        │
│ python→python │ twosum_hash_pointers        │ twosum_pointers → twosum_hash │ SRC / DST       │ SRC     │ SRC / DST → SRC             │ SRC        │
│ python→python │ palindrome_reverse_pointers │ pal_reverse → pal_pointers    │ SRC / DST       │ SRC     │ SRC / DST → SRC             │ SRC        │
│ python→python │ palindrome_reverse_pointers │ pal_pointers → pal_reverse    │ SRC / DST       │ DST     │ SRC / DST → SRC             │ DST        │
│ python→python │ toposort_kahn_dfs           │ kahn → topo_dfs               │ SRC / DST       │ SRC     │ SRC / DST → SRC             │ SRC        │
│ python→python │ toposort_kahn_dfs           │ topo_dfs → kahn               │ SRC / DST       │ SRC     │ SRC / DST → SRC             │ SRC        │
│ python→python │ lcs_dp_memo                 │ lcs_bottom_up → lcs_top_do

causal-shift : all 0.77 (74/96) · valid-only 0.77 (74/96) · valid rate 1.00

readout-shift: all 0.80 · readout-valid-only 0.84 (72/86) · readout valid rate 0.90

patched verdicts: {'SRC': 74, 'DST': 22} · patched readouts: {'neither': 10, 'SRC': 77, 'DST': 9} · kw agreement 
0.77

by language (causal · valid-only · readout): python→python 0.79 · 0.79 · 0.83 (n=24) · python→cpp 0.75 · 0.75 · 
0.75 (n=24) · cpp→python 0.79 · 0.79 · 0.83 (n=24) · cpp→cpp 0.75 · 0.75 · 0.79 (n=24)

by direction: a→b 0.71 · flipped 0.83

by pair (causal): bfs_dfs 8/8 · bsearch_leftmost 4/8 · merge_quick 8/8 · dijkstra_bfs 8/8 · prim_kruskal 8/8 · 
bubble_selection 6/8 · fib_rec_iter 4/8 · power_fast_naive 2/8 · twosum_hash_pointers 8/8 · 
palindrome_reverse_pointers 4/8 · toposort_kahn_dfs 6/8 · lcs_dp_memo 8/8

### Cross-language success rates

Per `(SRC → DST)` language pair: validity, causal-shift (all / valid-only), readout validity, readout-shift (all / readout-valid-only), and the patched verdicts; then a SRC × DST matrix of causal / readout-shift. Same oracle scoring as above.

In [11]:
# ── Cross-language success rates: one row per (SRC → DST) language pair, per config ──
LANG_LBL = {"python": "py", "cpp": "C++"}
lang_of = lambda r: (r.get("src_lang", "python"), r.get("dst_lang", "python"))

for cfg, rows in SWEEP.items():
    t = Table(title=f"Cross-language success rates · {cfg}", show_lines=False)
    for c, j in (("SRC → DST", "left"), ("n", "right"), ("valid", "right"), ("causal-shift", "right"),
                 ("causal | valid", "right"), ("readout valid", "right"), ("readout-shift", "right"),
                 ("readout | valid", "right"), ("patched verdicts", "left")):
        t.add_column(c, justify=j)
    groups = {lp: [r for r in rows if lang_of(r) == lp] for lp in dict.fromkeys(lang_of(r) for r in rows)}
    groups["all"] = rows
    for lp, rs in groups.items():
        v, rv = [r for r in rs if r["valid"]], [r for r in rs if r["ro_valid"]]
        name = "all" if lp == "all" else f"{LANG_LBL[lp[0]]} → {LANG_LBL[lp[1]]}"
        t.add_row(Text(name, style="bold" if lp == "all" else ""), str(len(rs)),
                  f"{len(v) / len(rs):.2f}",
                  f"{rate(rs, lambda r: r['shifted']):.2f} ({sum(r['shifted'] for r in rs)})",
                  f"{rate(v, lambda r: r['shifted']):.2f} ({sum(r['shifted'] for r in v)}/{len(v)})",
                  f"{len(rv) / len(rs):.2f}",
                  f"{rate(rs, lambda r: r['ro_shifted']):.2f} ({sum(r['ro_shifted'] for r in rs)})",
                  f"{rate(rv, lambda r: r['ro_shifted']):.2f} ({sum(r['ro_shifted'] for r in rv)}/{len(rv)})",
                  " ".join(f"{k}:{n}" for k, n in collections.Counter(r["v_patched"] for r in rs).most_common()),
                  end_section=(lp == list(groups)[-2]))
    Console(width=180).print(t)

    # SRC (rows) × DST (columns) matrix — causal-shift / readout-shift over all runs of the cell
    langs = [l for l in ("python", "cpp") if any(l in lang_of(r) for r in rows)]
    m = Table(title="causal-shift / readout-shift (rows = SRC, columns = DST)", show_lines=True)
    m.add_column("SRC \\ DST")
    for d in langs:
        m.add_column(LANG_LBL[d], justify="center")
    for s in langs:
        cells = []
        for d in langs:
            rs = groups.get((s, d), [])
            cells.append(f"{rate(rs, lambda r: r['shifted']):.2f} / {rate(rs, lambda r: r['ro_shifted']):.2f}  (n={len(rs)})"
                         if rs else "—")
        m.add_row(LANG_LBL[s], *cells)
    Console(width=120).print(m)

                                Cross-language success rates · span=name · V×4 · α=1.0 · I2=off                                
┏━━━━━━━━━━━┳━━━━┳━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ SRC → DST ┃  n ┃ valid ┃ causal-shift ┃ causal | valid ┃ readout valid ┃ readout-shift ┃ readout | valid ┃ patched verdicts ┃
┡━━━━━━━━━━━╇━━━━╇━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ py → py   │ 24 │  1.00 │    0.79 (19) │   0.79 (19/24) │          0.88 │     0.83 (20) │    0.86 (18/21) │ SRC:19 DST:5     │
│ py → C++  │ 24 │  1.00 │    0.75 (18) │   0.75 (18/24) │          0.92 │     0.75 (18) │    0.77 (17/22) │ SRC:18 DST:6     │
│ C++ → py  │ 24 │  1.00 │    0.79 (19) │   0.79 (19/24) │          0.88 │     0.83 (20) │    0.90 (19/21) │ SRC:19 DST:5     │
│ C++ → C++ │ 24 │  1.00 │    0.75 (18) │   0.75 (18/24) │          0.92 │     0.79 (19) │    0.82 (18/22) │ SRC:18 DST:6     │
├───────────┼────┼───────┼──────────────┼────────────────┼───────────────┼───────────────┼─────────────────┼──────────────────┤
│ all       │ 96 │  1.00 │    0.77 (74) │   0.77 (74/96) │          0.90 │     0.80 (77) │    0.84 (72/86) │ SRC:74 DST:22    │
└───────────┴────┴───────┴──────────────┴────────────────┴───────────────┴───────────────┴─────────────────┴──────────────────┘

causal-shift / readout-shift (rows = SRC, columns = DST) 
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ SRC \ DST ┃         py          ┃         C++         ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ py        │ 0.79 / 0.83  (n=24) │ 0.75 / 0.75  (n=24) │
├───────────┼─────────────────────┼─────────────────────┤
│ C++       │ 0.79 / 0.83  (n=24) │ 0.75 / 0.79  (n=24) │
└───────────┴─────────────────────┴─────────────────────┘

### Paper table — `tab:kv-goal`

One row per task direction (SRC task → DST task), one column per language direction. Each cell is the oracle verdict on
the patched run: \textsc{src} (it implements the source task, a success; dark green), \textsc{dst} (no-op), \textsc{both},
\textsc{neither} (gray). The last row is the causal-shift per language direction. Shown here as a rich table; the LaTeX
goes to `notebooks/Figures/transfer/kv_goal_table.tex`.

In [12]:
# ── paper table (tab:kv-goal): task direction × language direction, oracle verdict per cell + causal-shift row ──
TABLE_CFG = CFG                                   # which SWEEP config to tabulate (default: the last sweep's)
TABLE_TEX = os.path.join(env_utils.PROJECT_ROOT, "notebooks", "Figures", "transfer", "kv_goal_table.tex")
DIRS = [("python", "python"), ("python", "cpp"), ("cpp", "python"), ("cpp", "cpp")]
DIR_TEX = {("python", "python"): r"py$\to$py", ("python", "cpp"): r"py$\to$cpp",
           ("cpp", "python"): r"cpp$\to$py", ("cpp", "cpp"): r"cpp$\to$cpp"}
ALG_TEX = {   # short display names of the algorithm ids in PAIRS
    "bfs": "BFS", "dfs": "DFS", "binary_search": "binary search", "leftmost": "leftmost binary search",
    "merge_sort": "merge sort", "quick_sort": "quick sort", "dijkstra": "Dijkstra", "bfs_shortest": "BFS shortest path",
    "prim": "Prim's MST", "kruskal": "Kruskal's MST", "bubble_sort": "bubble sort", "selection_sort": "selection sort",
    "fib_recursive": "recursive Fibonacci", "fib_iterative": "iterative Fibonacci", "fast_power": "fast power",
    "naive_power": "naive power", "twosum_hash": "two-sum (hash map)", "twosum_pointers": "two-sum (two pointers)",
    "pal_reverse": "palindrome (reverse)", "pal_pointers": "palindrome (two pointers)", "kahn": "Kahn's toposort",
    "topo_dfs": "DFS toposort", "lcs_bottom_up": "LCS (bottom-up DP)", "lcs_top_down": "LCS (memoized)"}
V_TEX = {"SRC": r"\textcolor{ForestGreen}{\textsc{src}}", "DST": r"\textcolor{gray}{\textsc{dst}}",
         "both": r"\textcolor{gray}{\textsc{both}}", "neither": r"\textcolor{gray}{\textsc{neither}}",
         "parse_error": r"\textcolor{gray}{---}"}
V_RICH = {"SRC": ("SRC", "bold green4"), "DST": ("DST", "grey50"), "both": ("BOTH", "grey50"),
          "neither": ("NEITHER", "grey50"), "parse_error": ("—", "grey50")}

rows = SWEEP[TABLE_CFG]
lang_of = lambda r: (r.get("src_lang", "python"), r.get("dst_lang", "python"))
cell = {(r["src"], r["dst"], lang_of(r)): r["v_patched"] for r in rows}
tasks = list(dict.fromkeys((r["src"], r["dst"]) for r in rows))            # PAIRS order, a→b then b→a
dirs = [d for d in DIRS if any(lang_of(r) == d for r in rows)]
shift = {d: [r["shifted"] for r in rows if lang_of(r) == d] for d in dirs}
allshift = [r["shifted"] for r in rows if lang_of(r) in dirs]

# ── shown in the notebook ──
t = Table(title=f"tab:kv-goal · {TABLE_CFG} · oracle verdict on the patched run", show_lines=False)
t.add_column("SRC task → DST task")
for d in dirs:
    t.add_column(DIR_TEX[d].replace("$\\to$", " → "), justify="center")
for s, d_ in tasks:
    t.add_row(f"{ALG_TEX.get(s, s)} → {ALG_TEX.get(d_, d_)}",
              *(Text(*V_RICH[cell[(s, d_, d)]]) if (s, d_, d) in cell else Text("—", "grey50") for d in dirs))
t.add_row(Text("causal-shift", style="bold"),
          *(Text(f"{np.mean(shift[d]):.2f}", style="bold") for d in dirs), end_section=False)
Console(width=140).print(t)
print(f"overall causal-shift {np.mean(allshift):.3f} ({sum(allshift)}/{len(allshift)})")

# ── LaTeX (needs xcolor with dvipsnames for ForestGreen; booktabs) ──
tex = [r"\begin{table}[h!]", r"\centering", r"\small", r"\setlength{\tabcolsep}{4pt}", r"\begin{tabular}{l" + "c" * len(dirs) + "}", r"\toprule",
       r"SRC task $\to$ DST task & " + " & ".join(DIR_TEX[d] for d in dirs) + r" \\", r"\midrule"]
for s, d_ in tasks:
    tex.append(f"{ALG_TEX.get(s, s)} $\\to$ {ALG_TEX.get(d_, d_)} & "
               + " & ".join(V_TEX[cell[(s, d_, d)]] if (s, d_, d) in cell else "---" for d in dirs) + r" \\")
tex += [r"\midrule", r"\textbf{causal-shift} & " + " & ".join(f"\\textbf{{{np.mean(shift[d]):.2f}}}" for d in dirs) + r" \\",
        r"\bottomrule", r"\end{tabular}",
        (r"\caption{\small KV transfer on the broader goal (the next operation is not specified). Each cell is the oracle "
         r"verdict on the patched run: \textcolor{ForestGreen}{\textsc{src}} = implements the source task (success), "
         r"\textsc{dst} = implements its own task (no-op), \textsc{both} / \textsc{neither}. 250 goal heads read the "
         r"source task span (the algorithm name) from the pivot onwards, values $\times 4$. Overall causal-shift "
         f"${np.mean(allshift):.2f}$ (${sum(allshift)}/{len(allshift)}$)." + "}"),
        r"\label{tab:kv-goal}", r"\end{table}"]
TEX = "\n".join(tex)
os.makedirs(os.path.dirname(TABLE_TEX), exist_ok=True)
open(TABLE_TEX, "w").write(TEX + "\n")
print(f"\nLaTeX -> {TABLE_TEX}\n")
print(TEX)

       tab:kv-goal · span=name · V×4 · α=1.0 · I2=off · oracle verdict on the patched run       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┓
┃ SRC task → DST task                              ┃ py → py ┃ py → cpp ┃ cpp → py ┃ cpp → cpp ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━┩
│ BFS → DFS                                        │   SRC   │   SRC    │   SRC    │    SRC    │
│ DFS → BFS                                        │   SRC   │   SRC    │   SRC    │    SRC    │
│ binary search → leftmost binary search           │   DST   │   DST    │   DST    │    DST    │
│ leftmost binary search → binary search           │   SRC   │   SRC    │   SRC    │    SRC    │
│ merge sort → quick sort                          │   SRC   │   SRC    │   SRC    │    SRC    │
│ quick sort → merge sort                          │   SRC   │   SRC    │   SRC    │    SRC    │
│ Dijkstra → BFS shortest path                     │   SRC   │   SRC    │   SRC    │    SRC    │
│ BFS shortest path → Dijkstra                     │   SRC   │   SRC    │   SRC    │    SRC    │
│ Prim's MST → Kruskal's MST                       │   SRC   │   SRC    │   SRC    │    SRC    │
│ Kruskal's MST → Prim's MST                       │   SRC   │   SRC    │   SRC    │    SRC    │
│ bubble sort → selection sort                     │   SRC   │   DST    │   SRC    │    DST    │
│ selection sort → bubble sort                     │   SRC   │   SRC    │   SRC    │    SRC    │
│ recursive Fibonacci → iterative Fibonacci        │   DST   │   DST    │   DST    │    DST    │
│ iterative Fibonacci → recursive Fibonacci        │   SRC   │   SRC    │   SRC    │    SRC    │
│ fast power → naive power                         │   DST   │   SRC    │   DST    │    SRC    │
│ naive power → fast power                         │   DST   │   DST    │   DST    │    DST    │
│ two-sum (hash map) → two-sum (two pointers)      │   SRC   │   SRC    │   SRC    │    SRC    │
│ two-sum (two pointers) → two-sum (hash map)      │   SRC   │   SRC    │   SRC    │    SRC    │
│ palindrome (reverse) → palindrome (two pointers) │   SRC   │   DST    │   SRC    │    DST    │
│ palindrome (two pointers) → palindrome (reverse) │   DST   │   SRC    │   DST    │    SRC    │
│ Kahn's toposort → DFS toposort                   │   SRC   │   SRC    │   SRC    │    SRC    │
│ DFS toposort → Kahn's toposort                   │   SRC   │   DST    │   SRC    │    DST    │
│ LCS (bottom-up DP) → LCS (memoized)              │   SRC   │   SRC    │   SRC    │    SRC    │
│ LCS (memoized) → LCS (bottom-up DP)              │   SRC   │   SRC    │   SRC    │    SRC    │
│ causal-shift                                     │  0.79   │   0.75   │   0.79   │   0.75    │
└──────────────────────────────────────────────────┴─────────┴──────────┴──────────┴───────────┘

overall causal-shift 0.771 (74/96)

LaTeX -> /disk/u/arnab/Codes/Projects/goals/notebooks/Figures/transfer/kv_goal_table.tex

\begin{table}[h!]
\centering
\small
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lcccc}
\toprule
SRC task $\to$ DST task & py$\to$py & py$\to$cpp & cpp$\to$py & cpp$\to$cpp \\
\midrule
BFS $\to$ DFS & \textcolor{ForestGreen}{\textsc{src}} & \textcolor{ForestGreen}{\textsc{src}} & \textcolor{ForestGreen}{\textsc{src}} & \textcolor{ForestGreen}{\textsc{src}} \\
DFS $\to$ BFS & \textcolor{ForestGreen}{\textsc{src}} & \textcolor{ForestGreen}{\textsc{src}} & \textcolor{ForestGreen}{\textsc{src}} & \textcolor{ForestGreen}{\textsc{src}} \\
binary search $\to$ leftmost binary search & \textcolor{gray}{\textsc{dst}} & \textcolor{gray}{\textsc{dst}} & \textcolor{gray}{\textsc{dst}} & \textcolor{gray}{\textsc{dst}} \\
leftmost binary search $\to$ binary search & \textcolor{ForestGreen}{\textsc{src}} & \textcolor{ForestGreen}{\textsc{src}} & \textcolor{ForestGreen}{\textsc{s